# 05 — Baselines

Evaluate Persistence and Linear Regression with the shared prepared sessions. Run before notebook 06.


In [ ]:
%pip -q install numpy pandas scikit-learn joblib


In [ ]:
from pathlib import Path
import sys
import importlib
from urllib.request import urlopen

NOTEBOOK_DIR = Path.cwd() / "KC_CO2_Modular_Notebooks"
if not (NOTEBOOK_DIR / "kc_common.py").is_file():
    NOTEBOOK_DIR = Path.cwd()
REPO_REF = "6a1d9f2ce1559750832d108cb7d2b77db6501a79"
helper_path = NOTEBOOK_DIR / "kc_common.py"
if not helper_path.is_file():
    helper_url = f"https://raw.githubusercontent.com/Namelessr679/ESP32-Indoor-Air-Quality-1.0Monitor/{REPO_REF}/KC_CO2_Modular_Notebooks/kc_common.py"
    with urlopen(helper_url, timeout=30) as response:
        helper_source = response.read()
    compile(helper_source, "kc_common.py", "exec")
    if b"def prepare_experiment(" not in helper_source:
        raise RuntimeError("The downloaded helper is incompatible with this notebook revision.")
    NOTEBOOK_DIR.mkdir(parents=True, exist_ok=True)
    helper_path.write_bytes(helper_source)
sys.path.insert(0, str(NOTEBOOK_DIR))
import kc_common
if Path(kc_common.__file__).resolve() != helper_path.resolve():
    raise RuntimeError("A different kc_common module is already loaded. Restart the runtime and rerun setup.")
importlib.reload(kc_common)
from kc_common import experiment_run_dir, load_prepared


In [ ]:
from google.colab import drive

drive.mount("/content/drive")
PROJECT = Path("/content/drive/MyDrive/KC_CO2_Modular_Notebooks")
RUN_DIR = experiment_run_dir(PROJECT)


## Baseline implementations


In [ ]:
from __future__ import annotations

import json

from pathlib import Path

import numpy as np

import pandas as pd

from sklearn.linear_model import LinearRegression

from kc_common import (  # noqa: E402
    MODEL_FEATURES,
    SETUPS,
    forecast_future_co2,
    load_prepared,
    make_preprocessor,
    metrics_from_future_co2,
    model_output_to_future_co2,
    setup_key,
)

AVAILABLE_METHODS = ("Persistence", "Linear Regression")

RUN_NAME = "baseline_pipeline"

def _fit_linear_regression(config: dict) -> tuple[LinearRegression, object]:
    preprocessor = make_preprocessor(scale_numeric=False)
    train = config["train_df"]
    features = preprocessor.fit_transform(train[MODEL_FEATURES])
    model = LinearRegression()
    model.fit(features, train["target_change"].to_numpy())
    return model, preprocessor


def _fit_final_linear_regression(config: dict) -> tuple[LinearRegression, object]:
    preprocessor = make_preprocessor(scale_numeric=False)
    training = pd.concat(
        [config["train_df"], config["validation_df"]],
        ignore_index=True,
    )
    features = preprocessor.fit_transform(training[MODEL_FEATURES])
    model = LinearRegression()
    model.fit(features, training["target_change"].to_numpy())
    return model, preprocessor


def _predict(
    method: str,
    model: LinearRegression | None,
    preprocessor: object | None,
    frame: pd.DataFrame,
    config: dict,
    histories: dict,
) -> np.ndarray:
    if method == "Persistence":
        return frame["CO2_now"].to_numpy(dtype=float)
    if method != "Linear Regression" or model is None or preprocessor is None:
        raise ValueError(f"Unsupported baseline method: {method}")
    if config["approach"] == "direct":
        changes = model.predict(preprocessor.transform(frame[MODEL_FEATURES]))
        return model_output_to_future_co2(frame, changes)
    return forecast_future_co2(
        model,
        preprocessor,
        frame,
        config["horizon"],
        config["approach"],
        histories,
    )


def _prediction_rows(
    method: str,
    split: str,
    config: dict,
    frame: pd.DataFrame,
    predicted: np.ndarray,
) -> pd.DataFrame:
    rows = frame[["session_id", "time", "Actual"]].copy()
    rows["Predicted"] = np.asarray(predicted, dtype=float)
    rows["Model"] = method
    rows["Split"] = split
    rows["Horizon_min"] = config["horizon"]
    rows["Approach"] = config["approach"]
    rows["Run"] = RUN_NAME
    return rows


def run(run_dir: Path, output_dir: Path | None = None) -> dict[str, object]:
    """Evaluate available baselines using the existing prepared handoff."""
    prepared = load_prepared(run_dir)
    output_dir = output_dir or run_dir / "baseline_results"
    output_dir.mkdir(parents=True, exist_ok=True)
    histories = prepared["history_by_session"]
    metric_rows: list[dict[str, object]] = []
    prediction_frames: list[pd.DataFrame] = []

    for key in (setup_key(*setup) for setup in SETUPS):
        config = prepared["configurations"][key]
        validation = config["validation_df"]
        test = config["test_df"]
        for method in AVAILABLE_METHODS:
            if method == "Linear Regression":
                validation_model, validation_preprocessor = _fit_linear_regression(config)
                final_model, final_preprocessor = _fit_final_linear_regression(config)
            else:
                validation_model = final_model = None
                validation_preprocessor = final_preprocessor = None

            validation_prediction = _predict(
                method, validation_model, validation_preprocessor,
                validation, config, histories,
            )
            test_prediction = _predict(
                method, final_model, final_preprocessor,
                test, config, histories,
            )
            for split, frame, prediction in (
                ("Validation", validation, validation_prediction),
                ("Test", test, test_prediction),
            ):
                metrics = metrics_from_future_co2(frame["Actual"], prediction)
                metric_rows.append({
                    "Horizon_min": config["horizon"],
                    "Approach": config["approach"],
                    "Model": method,
                    "Split": split,
                    "Run": RUN_NAME,
                    **metrics,
                })
                prediction_frames.append(
                    _prediction_rows(method, split, config, frame, prediction)
                )

    metrics_frame = pd.DataFrame(metric_rows)
    predictions_frame = pd.concat(prediction_frames, ignore_index=True)
    metrics_frame.to_csv(output_dir / "baseline_metrics.csv", index=False)
    predictions_frame.to_csv(output_dir / "baseline_predictions.csv", index=False)
    pd.DataFrame([
        {"method": method, "status": "available"}
        for method in AVAILABLE_METHODS
    ]).to_csv(output_dir / "baseline_method_metadata.csv", index=False)
    (output_dir / "baseline_metadata.json").write_text(
        json.dumps({
            "run": RUN_NAME,
            "methods": list(AVAILABLE_METHODS),
            "setups": [setup_key(*setup) for setup in SETUPS],
            "source_run": str(run_dir),
        }, indent=2),
        encoding="utf-8",
    )
    return {
        "metadata": prepared["metadata"],
        "metrics": metrics_frame,
        "predictions": predictions_frame,
    }


## Evaluate and save results


In [ ]:
import joblib
from IPython.display import display

baseline_results = run(RUN_DIR)
result_path = RUN_DIR / "baseline_results" / "baseline_results.joblib"
joblib.dump(baseline_results, result_path, compress=3)
display(baseline_results["metrics"])
print("Saved baseline results:", result_path)
